# Estimating a probability density: histogram, kNN and KDE
A probability density function (PDF) describes where observations concentrate. Probability is **area under the density**, not its height at one point.

We estimate the same unknown density in three ways. For this demonstration only, we know the true density and can compare it with the estimates.

**Requirements:** Python, NumPy, SciPy, Matplotlib and astroML. Run the cells from top to bottom. Change the parameters to explore the tradeoffs.

`samples` contains the observed data; `x` is the grid on which we evaluate and plot the estimates.

In [ ]:
# Run once if dependencies are missing, then restart the kernel if needed.
# %pip install numpy scipy matplotlib astroML
import sys
sys.path.insert(0, "../..")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 13.1 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 15.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 738.7/738.7 kB 19.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 12.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [astroML]m7/9 [scikit-learn]data]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from astroML.density_estimation import KNeighborsDensity

rng = np.random.default_rng(12)
N = 400
component = rng.random(N) < 0.65
samples = np.where(component, rng.normal(-1.3, 0.55, N), rng.normal(1.4, 0.8, N))
# Evaluation points: where we calculate and plot each density estimate.
# This grid is not training data and does not define histogram bin edges.
x = np.linspace(-5, 5, 1200)
true_pdf = 0.65 * norm.pdf(x, -1.3, 0.55) + 0.35 * norm.pdf(x, 1.4, 0.8)

plt.rcParams.update({'figure.figsize': (10, 3.4), 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 11})
def reference(ax):
    ax.plot(x, true_pdf, 'k--', lw=2, label='True PDF')
    ax.set(xlabel='Value', ylabel='Density', xlim=(-5, 5), ylim=(0, 0.7))
    ax.grid(alpha=0.15)


## 1. Histogram: fix the intervals, count the samples
Choose bin edges. In bin $j$, count $n_j$ observations and divide by the total sample count $N$ and bin width $\Delta_j$:
$$\hat p(x)=\frac{n_j}{N\Delta_j},\qquad x\text{ in bin }j.$$
Each bar's area is its observed probability fraction.

**Benefit:** simple, fast, and easy to update by increasing bin counts.

**Problem:** bin width, edge position and range affect the result. Narrow bins are noisy; wide bins hide structure. A restricted range can discard observations. With fixed edges, new out-of-range samples need explicit handling.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), sharey=True)
for ax, bins in zip(axes, (8, 25, 100)):
    ax.hist(samples, bins=np.linspace(-5, 5, bins + 1), density=True,
            color='steelblue', alpha=0.65, label='Histogram')
    reference(ax)
    ax.set_title(f'{bins} equal-width bins')
axes[0].legend()
plt.tight_layout()
plt.show()

# Warning: density=True renormalizes using only samples inside the range.
narrow_edges = np.linspace(-2, 2, 20)
kept = np.count_nonzero((samples >= -2) & (samples <= 2))
print(f'A range of [-2, 2] keeps {kept}/{N} samples; discarded samples change the estimated distribution.')

## 2. k-nearest neighbours: fix the count, adapt the interval
At each query point $x$, find the distance $r_k(x)$ to its $k$th nearest sample. The symmetric interval has width $2r_k(x)$ and contains approximately a fraction $k/N$ of the observations:
$$\hat p(x)=\frac{k}{N\,2r_k(x)}.$$
Dense regions give small intervals and high estimates; sparse regions give wide intervals and low estimates.

**Benefit:** no bin edges, bin widths or bin range; the interval adapts locally.

**Problem:** we still choose $k$. Small $k$ gives spikes; large $k$ smooths away detail. Exact evaluation needs stored samples (or a neighbour-search structure). It has no simple histogram-like count-only online update: new samples can change the neighbours of many query points. Incremental implementations are possible, but require more storage and computation.

The basic kNN formula is a **local density estimate**: it need not integrate to one over the full real line. For fixed $k$, its far tails decay as $1/|x|$. We do not silently normalize it over the plotting range.

**Implementation:** `astroML.density_estimation.KNeighborsDensity(method="simple", n_neighbors=k)` performs fitting and density evaluation. Its output is divided by $N$ to convert number density into the estimate above.

In [ ]:
def knn_pdf(grid, data, k):
    data = np.asarray(data).reshape(-1, 1)
    grid = np.asarray(grid).reshape(-1, 1)
    model = KNeighborsDensity(method="simple", n_neighbors=k)
    model.fit(data)
    # astroML returns number density k / V; divide by N for k / (N V).
    return model.eval(grid) / len(data)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), sharey=True)
for ax, k in zip(axes, (5, 25, 100)):
    ax.plot(x, knn_pdf(x, samples, k), color='darkorange', label='kNN estimate')
    reference(ax)
    ax.set_title(f'k = {k}')
axes[0].legend()
plt.tight_layout()
plt.show()

query, k = 0.0, 25
radius = np.partition(np.abs(samples - query), k - 1)[k - 1]
fig, ax = plt.subplots()
ax.plot(samples, np.zeros(N), '|', color='0.5', markersize=12)
ax.axvspan(query - radius, query + radius, alpha=0.25, color='darkorange', label='Neighbour interval')
ax.axvline(query, color='k', label='Query point')
ax.set(xlabel='Value', yticks=[], title=f'{k} neighbours in an interval of width {2 * radius:.2f}')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Kernel density estimation: average smooth contributions
Place a normalized kernel at each sample and average the kernels. With Gaussian kernels and bandwidth $h$:
$$\hat p(x)=\frac{1}{Nh}\sum_{i=1}^{N}K\!\left(\frac{x-x_i}{h}\right),\qquad K(u)=\frac{e^{-u^2/2}}{\sqrt{2\pi}}.$$
Each sample contributes a smooth bump with area $1/N$. Their sum has total area one.

**Benefit:** a smooth, normalized PDF without bin edges. For fixed bandwidth, values on a fixed grid can be updated as samples arrive.

**Problem:** bandwidth selection replaces bin-width selection. Small $h$ gives noise; large $h$ merges peaks. A single bandwidth may not suit both dense and sparse regions. Gaussian kernels also put density outside a bounded physical domain.

In [ ]:
def kde_pdf(grid, data, h):
    if h <= 0:
        raise ValueError('Bandwidth must be positive')
    return norm.pdf((np.asarray(grid)[:, None] - np.asarray(data)[None, :]) / h).mean(axis=1) / h

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), sharey=True)
for ax, h in zip(axes, (0.08, 0.3, 1.0)):
    ax.plot(x, kde_pdf(x, samples, h), color='seagreen', label='Gaussian KDE')
    reference(ax)
    ax.set_title(f'Bandwidth h = {h}')
axes[0].legend()
plt.tight_layout()
plt.show()

# Show how the kernels add up, using a small subset for clarity.
subset = samples[:12]
h = 0.3
fig, ax = plt.subplots()
contributions = norm.pdf((x[:, None] - subset[None, :]) / h) / (len(subset) * h)
ax.plot(x, contributions, color='seagreen', alpha=0.3, lw=1)
ax.plot(x, contributions.sum(axis=1), color='seagreen', lw=2.5, label='Sum of 12 contributions')
ax.plot(subset, np.zeros(len(subset)), '|', color='k', markersize=10)
ax.set(xlabel='Value', ylabel='Density', title='One kernel per sample; sum gives the KDE')
ax.legend()
plt.tight_layout()
plt.show()

## Online update: what must be remembered?
For KDE with **fixed** $h$, a new observation $x_{N+1}$ gives
$$\hat p_{N+1}(x)=\frac{N}{N+1}\hat p_N(x)+\frac{1}{(N+1)h}K\!\left(\frac{x-x_{N+1}}{h}\right).$$
On a fixed evaluation grid, only the current estimates and $N$ are needed. Changing $h$ later or evaluating new points generally requires the samples again. The following verifies that incremental and batch evaluation agree.

In [ ]:
h = 0.3
online = np.zeros_like(x)
for count, value in enumerate(samples, start=1):
    online += (norm.pdf((x - value) / h) / h - online) / count
batch = kde_pdf(x, samples, h)
np.testing.assert_allclose(online, batch, rtol=1e-12, atol=1e-14)
print(f'Maximum online vs batch difference: {np.max(np.abs(online - batch)):.2e}')

## Comparison
| Method | Fixed choice | Local estimate | Main limitation | Simple incremental update? |
|---|---|---|---|---|
| Histogram | Bin edges and range | Count / (N × bin width) | Edge and width sensitivity | Yes, with fixed bins and range handling |
| kNN | Neighbour count k | k / (N × neighbour-interval width) | Choice of k; sample storage; tails | No count-only update; neighbour structures can be updated |
| KDE | Kernel and bandwidth h | Average kernel contributions | Bandwidth sensitivity; boundary leakage | Yes, on a fixed grid with fixed h |

**Try:** change `N` and rerun. Which structures persist as more samples become available? Shift histogram edges, vary `k`, and vary `h` to see how tuning changes the estimate.

Implementation reference: [astroML KNeighborsDensity](https://www.astroml.org/modules/generated/astroML.density_estimation.KNeighborsDensity.html).